# Session 1 — The agent loop

**DAT 5668 · Module 1, Session 1 · Fundamentals (about 80 minutes)**

**Families this session: both.** An LLM-based agent decides by calling tools over language. An RL-based agent decides from a learned value over a numeric state. Same loop. Different machinery.

![The agent loop](../../assets/agent-loop.png)

![One loop, two agent families](../../assets/two-families.png)

## Run of show
- 10 min — In the room: what is an agent? Cold call, then the definition below. There is no recording to watch first.
- 20 min — One loop, two implementations, and the cost of the wrong family.
- 35 min — Live demo: the same refund request as an LLM-style agent and as an RL-style agent.
- 15 min — Managerial questions that Session 2's practice will use.

## Learning goals
After this session students can:
- State the loop **observe → decide → act → feedback**.
- Say when a step should stay with a person.
- Choose, with a business reason, an LLM-based agent, an RL-based agent, or neither.
- Name the design choices: goal, data, actions, feedback, human checkpoint.


## 1. One definition

An agent acts toward a **goal** in an **environment**.

| Step | Business meaning | Refund desk example |
| --- | --- | --- |
| Observe | Read the current situation | Ticket text, order age, customer tier |
| Decide | Pick the next action | Look up policy, approve, refuse, escalate |
| Act | Change something outside the model | Write a reply or open a case |
| Feedback | Learn whether the action helped | Customer accepted, policy violation, cost |

A chatbot that only writes a paragraph is not yet an agent. An agent is defined by the action and the feedback, not by the model brand.

**Wrong family, wrong job**
- A chat reply is a poor inventory controller: the reward is a number that repeats every day, and the state is stock, not a paragraph.
- A DQN is a poor contract reviewer: the "state" is unstructured language, and the action is a tool call plus a cited reason.


## 2. Worked case — damaged-item refund

Instructional example, not firm data. Policy the business will stand behind:

- Damaged items may be refunded within 30 days.
- Orders older than 30 days are refused unless a manager escalates.
- Refund amount is the order total, not a number the model invents.

We will run **the same three tickets** through both families.


## Optional model

The decide step is a **local stand-in**. It must call `lookup_policy`, then `lookup_amount`, then finish from those tool results. It does not invent a dollar amount.

If the environment variable `GEMINI_API_KEY` (or `GOOGLE_API_KEY`) is set, that same step asks Gemini which tool to call next. The tools still run in this notebook. A missing key, a refused call, or a response that names a tool outside the schema falls back to the stand-in, so class runs either way. Do not paste a firm secret into the ticket text.


In [ ]:
import json
import os
import urllib.request

# Shared tickets. Both agents see the same observations.
TICKETS = [
    {"id": "A", "text": "Item arrived cracked", "order_age_days": 12, "order_total": 80},
    {"id": "B", "text": "Changed my mind", "order_age_days": 10, "order_total": 40},
    {"id": "C", "text": "Screen is cracked", "order_age_days": 45, "order_total": 200},
]

# Tools an LLM-style agent is allowed to call. The model does not invent the policy.
POLICY = {
    "damaged": {"max_age_days": 30, "outcome": "refund"},
    "changed mind": {"max_age_days": 0, "outcome": "refuse"},
}


def lookup_policy(text):
    """Observe the rule that applies. This is a tool, not a guess."""
    lowered = text.lower()
    if "crack" in lowered:
        return POLICY["damaged"]
    if "changed my mind" in lowered:
        return POLICY["changed mind"]
    return None


# The tool schema is what a tool-calling LLM is allowed to choose. It cannot invent a fourth tool.
TOOL_SCHEMA = {
    "lookup_policy": "Read the refund rule for this ticket text.",
    "lookup_amount": "Read the order total from the system of record.",
    "finish": "Stop and return refund, refuse, or escalate.",
}


def local_decide(ticket, memory):
    """Local stand-in. Same contract as a tool-calling model: name the next tool, not a dollar amount."""
    called = [step["tool"] for step in memory]
    if "lookup_policy" not in called:
        return {"tool": "lookup_policy"}
    if "lookup_amount" not in called:
        return {"tool": "lookup_amount"}
    rule = next(step["result"] for step in memory if step["tool"] == "lookup_policy")
    if rule is None:
        action, reason = "escalate", "No policy tool matched the ticket."
    elif ticket["order_age_days"] <= rule["max_age_days"] and rule["outcome"] == "refund":
        action, reason = "refund", f"Policy allows a refund within {rule['max_age_days']} days."
    else:
        action, reason = "refuse", "Policy tool does not allow this refund."
    return {"tool": "finish", "action": action, "reason": reason}


def gemini_decide(ticket, memory):
    """Ask Gemini for the next tool. Returns None when no key is set or the call is unusable."""
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not key:
        return None
    model = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
    prompt = {
        "instruction": (
            "You are the decide step of a tool-using agent. "
            "Reply with JSON only: {\"tool\": <one name from schema>, "
            "\"action\": <refund|refuse|escalate, only if tool is finish>, "
            "\"reason\": <one sentence from the tool results>} . "
            "Do not invent a refund amount. Do not name a tool that is not in the schema."
        ),
        "schema": TOOL_SCHEMA,
        "ticket": {"id": ticket["id"], "text": ticket["text"], "order_age_days": ticket["order_age_days"]},
        "memory": memory,
    }
    body = json.dumps({
        "contents": [{"role": "user", "parts": [{"text": json.dumps(prompt)}]}],
        "generationConfig": {"temperature": 0, "responseMimeType": "application/json"},
    }).encode()
    request = urllib.request.Request(
        f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent",
        data=body,
        headers={"Content-Type": "application/json", "x-goog-api-key": key},
    )
    try:
        with urllib.request.urlopen(request, timeout=12) as response:
            payload = json.loads(response.read().decode())
        text = payload["candidates"][0]["content"]["parts"][0]["text"]
        decision = json.loads(text)
    except Exception:
        return None
    tool = decision.get("tool") if isinstance(decision, dict) else None
    if tool not in TOOL_SCHEMA:
        return None
    if tool != "finish":
        return {"tool": tool}
    action = decision.get("action")
    if action not in ("refund", "refuse", "escalate"):
        return None
    reason = decision.get("reason") or "Finished from the tool results."
    return {"tool": "finish", "action": action, "reason": reason}


def llm_decide(ticket, memory):
    """Gemini when a key is present and the reply is valid; otherwise the local stand-in."""
    remote = gemini_decide(ticket, memory)
    if remote is not None:
        llm_decide.source = "gemini"
        return remote
    llm_decide.source = "local stand-in"
    return local_decide(ticket, memory)


def run_tool(tool, ticket):
    if tool == "lookup_policy":
        return lookup_policy(ticket["text"])
    if tool == "lookup_amount":
        return ticket["order_total"]
    raise ValueError(tool)


def llm_style_agent(ticket):
    """LLM-based agent: observe, choose a tool, act, store the result, repeat."""
    memory = []
    for _ in range(4):
        decision = llm_decide(ticket, memory)
        if decision["tool"] == "finish":
            amount = ticket["order_total"] if decision["action"] == "refund" else 0
            return {
                "action": decision["action"],
                "reason": decision["reason"],
                "steps": [step["tool"] for step in memory],
                "feedback": {"auditable": True, "amount": amount},
            }
        memory.append({"tool": decision["tool"], "result": run_tool(decision["tool"], ticket)})
    return {"action": "escalate", "reason": "Agent exceeded its step budget.", "steps": [], "feedback": {"auditable": True, "amount": 0}}


print("Tool schema the LLM-based agent may call:", list(TOOL_SCHEMA))
_key_set = bool(os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"))
print(
    "Decide step:",
    "Gemini when the call succeeds; otherwise the local stand-in" if _key_set else "local stand-in",
)
for ticket in TICKETS:
    result = llm_style_agent(ticket)
    print(
        f"Ticket {ticket['id']}: steps={result['steps']} -> {result['action']:8} | "
        f"{result['reason']} | amount={result['feedback']['amount']}"
    )


## 3. The same loop as an RL-based agent

An RL-based agent does not read the sentence and does not call those tools. Someone has already mapped the ticket to a **state**. The agent picks an **action** because that action has the best learned value. Feedback is a **reward**, not a citation. Sessions 4 through 10 train this family. Sessions 1 and 3 train the LLM-based family. Sessions 11 and 12 combine them.

State used here (instructional, deliberately small):

- `fresh_damage` — damaged and inside the window
- `stale_damage` — damaged and outside the window
- `other` — anything else

Actions: `refund`, `refuse`, `escalate`.


In [ ]:
# A Q-table after learning. Session 1 shows the table; Session 8 trains one.
# Values are expected long-run reward, not probabilities.
Q = {
    "fresh_damage": {"refund": 8.0, "refuse": -2.0, "escalate": 1.0},
    "stale_damage": {"refund": -6.0, "refuse": 3.0, "escalate": 2.0},
    "other": {"refund": -4.0, "refuse": 2.0, "escalate": 2.5},
}


def to_state(ticket):
    damaged = "crack" in ticket["text"].lower()
    if damaged and ticket["order_age_days"] <= 30:
        return "fresh_damage"
    if damaged:
        return "stale_damage"
    return "other"


def rl_style_agent(ticket):
    state = to_state(ticket)
    # Decide: greedy on the learned values. Ties break toward the first max.
    action = max(Q[state], key=Q[state].get)
    reward = Q[state][action]
    return {"observation": state, "action": action, "value": reward}


print("RL-style policy on the same tickets")
for ticket in TICKETS:
    result = rl_style_agent(ticket)
    print(f"Ticket {ticket['id']}: state={result['observation']:14} action={result['action']:8} value={result['value']}")


## 4. What to say to a manager

Ask these five questions before anyone builds the agent. They are the same questions for both families.

1. **Goal** — What decision, if automated, would change a KPI this quarter?
2. **Data** — Is the input a document/ticket, or a repeated numeric state?
3. **Actions** — What may the system do, and what is forbidden?
4. **Feedback** — Is the signal a citation and a human rating, or a numeric reward every step?
5. **Human checkpoint** — Which action is too costly to take without a person?

| Ticket | LLM-style action | RL-style action | Checkpoint |
| --- | --- | --- | --- |
| A, cracked, day 12 | refund, with the policy tool | refund | Auto-approve inside policy |
| B, changed mind | refuse | escalate (`other` has no damage rule) | A person should see "other" |
| C, cracked, day 45 | refuse | refuse | Do not let either family override the window |

The families can disagree (ticket B). That disagreement is a design fact, not a bug. Session 2. You will build the tool-using side and see what fails if the tool is skipped.

## Bridge to Session 2
Students will replace a reply-only chatbot with an agent that must call `lookup_policy`, `lookup_order`, and `open_case`.


## Fill this with me

Say the two actions out loud before you run the cell. Then change the age to 60 and run it again. Then change the text to "Changed my mind" and watch the families disagree.


In [ ]:
# FILL WITH THE CLASS. refund, refuse, or escalate.
guess_llm = ""
guess_rl = ""

ticket_d = {"id": "D", "text": "Lid arrived cracked", "order_age_days": 20, "order_total": 50}
llm = llm_style_agent(ticket_d)
rl = rl_style_agent(ticket_d)
print("LLM-based agent:", llm["action"], "| you said:", guess_llm or "(type it first)")
print("RL-based agent: ", rl["action"], "| you said:", guess_rl or "(type it first)")
print("Edit order_age_days or the text above, run again, and tell the room what moved.")
